### Dependencies

In [6]:
from apify_client import ApifyClient
from dotenv import load_dotenv
import pandas as pd
import re
import requests
import json
import os
import shutil
from datetime import datetime
from pathlib import Path
from apify_class import Apify

## Save Profile Picture

In [ ]:
import requests

url = "https://instagram.fisb17-1.fna.fbcdn.net/v/t51.82787-19/525781976_18071951138078352_5065639200253104830_n.jpg?efg=eyJ2ZW5jb2RlX3RhZyI6InByb2ZpbGVfcGljLmRqYW5nby4xMDgwLmMyIn0&_nc_ht=instagram.fisb17-1.fna.fbcdn.net&_nc_cat=100&_nc_oc=Q6cZ2gEVYdsLSF2tU9vYFPcHw_iXoZX9-klVLnAKky0N5CuNcq-nVZXbi2OhBIAtMSffUm8&_nc_ohc=nlBRVVj9-x8Q7kNvwE5CGHZ&_nc_gid=82PAI0sd7oaW1dRclrXGhg&edm=AP4sbd4BAAAA&ccb=7-5&oh=00_AfyLNdyV5AJgXBsYec5Tov01ZXgdONbQTpzisQoqrZfykQ&oe=69C87744&_nc_sid=7a9f4b"
# Send request
response = requests.get(url)
# Check if download was successful
if response.status_code == 200:
    with open("../data/_saras.archives/_saras.archives.jpg", "wb") as f:
        f.write(response.content)

In [32]:
import boto3
import requests
import os
from dotenv import load_dotenv

load_dotenv()

s3 = boto3.client(
    's3',
    aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY'),
    region_name=os.getenv('AWS_REGION')
)

BUCKET = os.getenv('AWS_S3_BUCKET')

def upload_profile_pic(username: str, image_url: str) -> str:
    """
    Downloads profile pic from Instagram URL,
    uploads to S3, returns the public URL.
    """
    # Download the image
    response = requests.get(image_url)
    if response.status_code != 200:
        return None

    # Upload to S3
    key = f"brand-profile-pics/{username}.jpg" ## CHECK THIS BEFORE EXECUTION
    s3.put_object(
        Bucket=BUCKET,
        Key=key,
        Body=response.content,
        ContentType='image/jpeg'
    )

    # Return the public URL
    public_url = f"https://{BUCKET}.s3.{os.getenv('AWS_REGION')}.amazonaws.com/{key}"
    return public_url

url = upload_profile_pic("elo.shoponline", "https://instagram.fisb1-2.fna.fbcdn.net/v/t51.82787-19/642521441_18565058239031940_5851832043474388085_n.jpg?efg=eyJ2ZW5jb2RlX3RhZyI6InByb2ZpbGVfcGljLmRqYW5nby4xMDgwLmMyIn0&_nc_ht=instagram.fisb1-2.fna.fbcdn.net&_nc_cat=1&_nc_oc=Q6cZ2gGEW2u4ZukWCWP-ZsH9KNWr-QAtjFv-JM-J0ispag5PRTwBjpVsAPmQhUdrpmI1xj0&_nc_ohc=SfqKA60tATIQ7kNvwFG0J6f&_nc_gid=ecuvgGj9SZvXGhWXflK2Mw&edm=ALGbJPMBAAAA&ccb=7-5&oh=00_Af3zaKizzpJj0_cONhfr0uzucF2dMUmWGvJHNc8lI43fsw&oe=69D43586&_nc_sid=7d3ac5")
print(url)

https://upclout-profile-pics.s3.us-west-1.amazonaws.com/brand-profile-pics/elo.shoponline.jpg


## Influencer -> AWS

In [ ]:
import boto3
import os
from dotenv import load_dotenv

load_dotenv()

s3 = boto3.client(
    's3',
    aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY'),
    region_name=os.getenv('AWS_REGION')
)

BUCKET = os.getenv('AWS_S3_BUCKET')
REGION = os.getenv('AWS_REGION')

def upload_all_profile_pics(base_path='../data/influencers_DONE'):
    uploaded = 0
    failed = []

    for username in os.listdir(base_path):
        folder = os.path.join(base_path, username)
        if not os.path.isdir(folder):
            continue

        jpg_path = os.path.join(folder, f"{username}.jpg")
        if not os.path.exists(jpg_path):
            failed.append((username, "No .jpg found"))
            continue

        try:
            key = f"profile-pics/{username}.jpg"
            s3.upload_file(
                jpg_path,
                BUCKET,
                key,
                ExtraArgs={'ContentType': 'image/jpeg'}
            )
            uploaded += 1
            print(f"✅ [{uploaded}] {username}")
        except Exception as e:
            failed.append((username, str(e)))
            print(f"❌ {username} — {e}")

    print(f"\nDone! Uploaded: {uploaded} | Failed: {len(failed)}")
    if failed:
        print("Failed uploads:")
        for name, reason in failed:
            print(f"  - {name}: {reason}")

    return uploaded, failed

upload_all_profile_pics()

## Brand -> AWS

In [ ]:
import boto3
import os
from dotenv import load_dotenv

load_dotenv()

s3 = boto3.client(
    's3',
    aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY'),
    region_name=os.getenv('AWS_REGION')
)

BUCKET = os.getenv('AWS_S3_BUCKET')
REGION = os.getenv('AWS_REGION')

def upload_all_brand_pics(base_path='../data/brands_not_in_AWS'):
    uploaded = 0
    failed = []

    brands = get_brand_list()
    left_overs = []

    for username in os.listdir(base_path):
        folder = os.path.join(base_path, username)
        if not os.path.isdir(folder):
            continue

        jpg_path = os.path.join(folder, f"{username}.jpg")
        if not os.path.exists(jpg_path):
            failed.append((username, "No .jpg found"))
            continue

        try:
            # We use a separate folder prefix for brands
            key = f"brand-profile-pics/{username}.jpg"
            s3.upload_file(
                jpg_path,
                BUCKET,
                key,
                ExtraArgs={'ContentType': 'image/jpeg'}
            )
            uploaded += 1
            print(f"✅ [{uploaded}] Brand: {username}")
        except Exception as e:
            failed.append((username, str(e)))
            print(f"❌ {username} — {e}")

    print(f"\nDone! Uploaded Brands: {uploaded} | Failed: {len(failed)}")
    return uploaded, failed 

# To run it:
f, l = upload_all_brand_pics()
print(u)
print(f)

## Influencer Rec FOR BRANDS

In [ ]:
import json
import psycopg2

def load_recommendations_to_postgres(json_path='similarity_matches.json'):
    # 1. Connect to PostgreSQL
    conn = psycopg2.connect(database='postgres', user='postgres', password='1040')
    cur = conn.cursor()
    
    # 2. SQL query to create the table
    create_table_query = """
    CREATE TABLE IF NOT EXISTS brand_recommendations (
        id SERIAL PRIMARY KEY,
        brand_id BIGINT NOT NULL,
        recommended_influencer_id BIGINT NOT NULL,
        UNIQUE(brand_id, recommended_influencer_id)
    );
    """
    
    cur.execute(create_table_query)
    conn.commit()
    
    # 3. Load JSON file
    with open(json_path, 'r') as f:
        data = json.load(f)
        
    recommendations = data.get('top_influencers_for_brand', {})
    
    # 4. SQL Query to insert data (handling duplicates out of the box)
    insert_query = """
    INSERT INTO brand_recommendations (brand_id, recommended_influencer_id)
    VALUES (%s, %s)
    ON CONFLICT (brand_id, recommended_influencer_id) DO NOTHING;
    """
    
    count = 0
    # Loop over every brand's ID in the JSON dictionary
    for brand_id, influencers in recommendations.items():
        # Loop over the 10 recommended influencer IDs for this brand
        for influencer_id in influencers:
            cur.execute(insert_query, (int(brand_id), int(influencer_id)))
            count += 1
            
    # Commit all your insertions and close connections
    conn.commit()
    cur.close()
    conn.close()
    
    print(f"Successfully loaded {count} recommendations into PostgreSQL!")

# Execute the function!
load_recommendations_to_postgres('similarity_matches.json')


Successfully loaded 11736 recommendations into PostgreSQL!


## Brand Rec FOR INFLUENCERS

In [ ]:
import json
import psycopg2

def load_recommendations_to_postgres(json_path='similarity_matches.json'):
    # 1. Connect to PostgreSQL
    conn = psycopg2.connect(database='postgres', user='postgres', password='1040')
    cur = conn.cursor()
    
    # 2. SQL query to create the table
    create_table_query = """
    CREATE TABLE IF NOT EXISTS influencer_recommendations (
        id SERIAL PRIMARY KEY,
        influencer_id BIGINT NOT NULL,
        recommended_brand_id BIGINT NOT NULL,
        UNIQUE(influencer_id, recommended_brand_id)
    );
    """
    
    cur.execute(create_table_query)
    conn.commit()
    
    # 3. Load JSON file
    with open(json_path, 'r') as f:
        data = json.load(f)
        
    recommendations = data.get('top_brands_for_influencer', {})
    
    # 4. SQL Query to insert data (handling duplicates out of the box)
    insert_query = """
    INSERT INTO influencer_recommendations (influencer_id, recommended_brand_id)
    VALUES (%s, %s)
    ON CONFLICT (influencer_id, recommended_brand_id) DO NOTHING;
    """
    
    count = 0
    # Loop over every brand's ID in the JSON dictionary
    for brand_id, influencers in recommendations.items():
        # Loop over the 10 recommended influencer IDs for this brand
        for influencer_id in influencers:
            cur.execute(insert_query, (int(brand_id), int(influencer_id)))
            count += 1
            
    # Commit all your insertions and close connections
    conn.commit()
    cur.close()
    conn.close()
    
    print(f"Successfully loaded {count} recommendations into PostgreSQL!")

# Execute the function!
load_recommendations_to_postgres('similarity_matches.json')

In [72]:
import json
import re

results = []

with open('../tmp/niche_dry_run_output.txt', 'r', encoding='utf-16') as f:
    content = f.read()

blocks = re.split(r'-{3,}', content)

for block in blocks:
    block = block.strip()
    if not block:
        continue

    username_match = re.search(r'Processing @(\S+)', block)
    prediction_match = re.search(r'AI PREDICTION: \[(.+?)\]', block)

    if username_match and prediction_match:
        results.append({
            "username": username_match.group(1),
            "new_cat": prediction_match.group(1)
        })

with open('output.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(json.dumps(results[:3], indent=2, ensure_ascii=False))  # preview first 3

[
  {
    "username": "elo.shoponline",
    "new_cat": "Clothing (Brand)"
  },
  {
    "username": "hafeez_13_22",
    "new_cat": "General Interest"
  },
  {
    "username": "thaan_lo",
    "new_cat": "Clothing store"
  }
]


In [73]:
import json
import psycopg2

def update_categories_from_json(json_file):
    with open(json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    conn = psycopg2.connect(database='postgres', user='postgres', password='1040')
    cur = conn.cursor()

    updated = 0
    not_found = 0

    for entry in data:
        username = entry['username']
        new_cat = entry['new_cat']

        cur.execute("""
            UPDATE brands
            SET businesscategoryname = %s
            WHERE username = %s
        """, (new_cat, username))

        if cur.rowcount == 0:
            print(f"No match found for username: {username}")
            not_found += 1
        else:
            updated += 1

    conn.commit()
    cur.close()
    conn.close()

    print(f"\nDone! Updated: {updated} | Not found: {not_found}")

update_categories_from_json('output.json')


Done! Updated: 23 | Not found: 0


In [ ]:
import json

def extract_titles_to_profiles(json_path='saved_posts.json', profiles_path='../insta_profiles.txt'):
    """Extract all titles from saved_posts.json and append them to insta_profiles.txt (no duplicates with existing entries)."""
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    titles = [post['title'] for post in data.get('saved_saved_media', []) if 'title' in post]

    # Read existing profiles to avoid duplicates
    with open(profiles_path, 'r', encoding='utf-8') as f:
        existing = set(line.strip() for line in f if line.strip())

    # Filter out titles already present
    new_titles = [t for t in titles if t not in existing]
    # Also deduplicate within the new titles while preserving order
    seen = set()
    unique_new = []
    for t in new_titles:
        if t not in seen:
            seen.add(t)
            unique_new.append(t)

    # Append to file
    with open(profiles_path, 'a', encoding='utf-8') as f:
        for title in unique_new:
            f.write(title + '\n')

    print(f"Added {len(unique_new)} new profiles (skipped {len(titles) - len(unique_new)} duplicates)")
    return unique_new

# Run it
extract_titles_to_profiles()


In [8]:
def remove_dup_influencers() -> None:

    with open("../insta_profiles.txt", "r") as file:
        influencers = [line.strip() for line in file.readlines()]

    size_with_duplicates: int = len(influencers)
    remove_duplicate_list = set(influencers)
    size_without_duplicates: int = len(remove_duplicate_list)

    if size_with_duplicates == size_without_duplicates:
        print("Already Up-to-date")
        return
    
    """with open("../insta_profiles.txt", "w") as file:
        for influencer in remove_duplicate_list:
            file.write(influencer + "\n")
    """
    print(f"Removed duplicates. {size_with_duplicates - size_without_duplicates} entries deleted.")
